# Generation of useful files

It is the first notebook to execute

# 1) Atlas datasets

In [21]:
from pprint import pprint
from random import shuffle

from scripts.utils.file_utils import load_json, dump_json
from scripts.ripe_atlas.atlas_api import get_atlas_anchors, get_atlas_probes
from default import USER_ANCHORS_FILE, USER_PROBES_FILE, USER_PROBES_AND_ANCHORS_FILE

# If you set reproducibility to True you should get the same datasets used in our work
# Set to False, if you want to run on today's data
reproducibility = False

## Retrieve atlas probes and anchors


In [22]:
# 把gen_dataset生成的探针和锚点拿过来用
from pathlib import Path
import shutil
DATA_DIR = Path("/home/lzj/geoloc-imc-2023/Geo_mycode/data/generate")


# 从 DATA_DIR / "user_anchors.json" 复制对应文件到 USER_ANCHORS_FILE

shutil.copy(DATA_DIR / "user_anchors.json", USER_ANCHORS_FILE)
shutil.copy(DATA_DIR / "user_probes.json", USER_PROBES_FILE)



PosixPath('/home/lzj/geoloc-imc-2023/datasets/user_datasets/atlas/user_probes.json')

These two files will be filtered and updated step by step during the execution of this notebook
这两个文件将在本笔记本的执行过程中逐步过滤和更新

### test loading

In [23]:
probes = load_json(USER_PROBES_FILE)
anchors = load_json(USER_ANCHORS_FILE)

print(f"selected probes: {len(probes)}")
print(f"selected anchors: {len(anchors)}")

for i, probe in enumerate(probes):
    if i > 10:
        break
    pprint(probe)

selected probes: 11
selected anchors: 844
{'address_v4': '147.189.137.155',
 'asn_v4': '64249',
 'city': 'Boston',
 'country_code': 'US',
 'geometry': {'coordinates': [42.3642, -71.0257], 'type': 'Point'},
 'id': 0,
 'is_anchor': False}
{'address_v4': '107.172.196.228',
 'asn_v4': '36352',
 'city': 'Buffalo',
 'country_code': 'US',
 'geometry': {'coordinates': [42.8865, -78.8784], 'type': 'Point'},
 'id': 1,
 'is_anchor': False}
{'address_v4': '172.245.234.221',
 'asn_v4': '136258',
 'city': 'Chicago',
 'country_code': 'US',
 'geometry': {'coordinates': [41.85, -87.65], 'type': 'Point'},
 'id': 2,
 'is_anchor': False}
{'address_v4': '45.14.113.135',
 'asn_v4': '40676',
 'city': 'Chicago',
 'country_code': 'US',
 'geometry': {'coordinates': [41.8713, -87.6277], 'type': 'Point'},
 'id': 3,
 'is_anchor': False}
{'address_v4': '78.142.8.146',
 'asn_v4': '6597',
 'city': 'Atlanta',
 'country_code': 'US',
 'geometry': {'coordinates': [33.749, -84.388], 'type': 'Point'},
 'id': 4,
 'is_anchor

In [24]:
probes_and_anchors = probes + anchors
print(len(probes_and_anchors))
shuffle(probes_and_anchors)

dump_json(probes_and_anchors, USER_PROBES_AND_ANCHORS_FILE)

855


# 2) Geographical datasets

In [25]:
from scripts.utils.file_utils import load_json, dump_json
from scripts.utils.helpers import distance
from default import COUNTRIES_TXT_FILE, COUNTRIES_JSON_FILE, USER_ANCHORS_FILE, USER_PROBES_FILE

## Get country dataset

In [26]:
countries = {}
with open(COUNTRIES_TXT_FILE, "r", encoding="utf8") as f:
    for i, row in enumerate(f.readlines()):

        row = [value.strip() for value in row.split(" ")]
        countries[row[0]] = {
            "latitude": row[1],
            "longitude": row[2],
            "name": row[3],
        }

for i, (country_code, geoloc) in enumerate(countries.items()):
    if i > 10:
        break
    print(f"{country_code} : {geoloc}")

# save results
dump_json(countries, COUNTRIES_JSON_FILE)

AD : {'latitude': '42.546245', 'longitude': '1.601554', 'name': 'Andorra'}
AE : {'latitude': '23.424076', 'longitude': '53.847818', 'name': 'United'}
AF : {'latitude': '33.93911', 'longitude': '67.709953', 'name': 'Afghanistan'}
AG : {'latitude': '17.060816', 'longitude': '-61.796428', 'name': 'Antigua'}
AI : {'latitude': '18.220554', 'longitude': '-63.068615', 'name': 'Anguilla'}
AL : {'latitude': '41.153332', 'longitude': '20.168331', 'name': 'Albania'}
AM : {'latitude': '40.069099', 'longitude': '45.038189', 'name': 'Armenia'}
AN : {'latitude': '12.226079', 'longitude': '-69.060087', 'name': 'Netherlands'}
AO : {'latitude': '-11.202692', 'longitude': '17.873887', 'name': 'Angola'}
AQ : {'latitude': '-75.250973', 'longitude': '-0.071389', 'name': 'Antarctica'}
AR : {'latitude': '-38.416097', 'longitude': '-63.616672', 'name': 'Argentina'}


# 3) Other various files

In [30]:
import math
import pickle
import radix
import requests
from copy import deepcopy

from multiprocessing import Pool

from scripts.utils.file_utils import load_json, dump_json
from scripts.utils.helpers import haversine
from scripts.analysis.analysis import compute_remove_wrongly_geolocated_probes, compute_rtts_per_dst_src
from default import *

DB_HOST = "localhost"
GEO_REPLICATION_DB = "geolocation_replication"
ANCHORS_MESHED_PING_TABLE = f"anchors_meshed_pings"
PROBES_TO_ANCHORS_PING_TABLE = f"ping_10k_to_anchors"

LIMIT = 1000

## Generate ip level target list for all /24 prefixes
生成所有/24前缀的IP级别目标列表

In [31]:
targets_per_prefix = {}

with open(ADDRESS_FILE, "r") as f:
    for i, row in enumerate(f.readlines()[1:]):
        row = row.split("\t")

        # get prefix from hex value
        prefix_hex = row[0]
        prefix = ["".join(x) for x in zip(*[iter(prefix_hex)]*2)]
        prefix = [int(x, 16) for x in prefix]
        prefix = ".".join(str(x) for x in prefix)

        target_list = row[-1].strip("\n")
        target_list = target_list.split(",")

        # parse and save targets
        if target_list[0] != '-':
            for i, target in enumerate(target_list):
                target_list[i] = prefix.split(".")[:-1]
                target_list[i].append(str(int(target, 16)))
                target_list[i] = ".".join(target_list[i])

            try:
                targets_per_prefix[prefix].extend(target_list)
            except KeyError:
                targets_per_prefix[prefix] = target_list

In [32]:
dump_json(targets_per_prefix, USER_HITLIST_FILE)

print("target hitlist")
for i, prefix in enumerate(targets_per_prefix):
    if i > 10:
        break
    print("prefix:", prefix, "target hitlist:", targets_per_prefix[prefix])

target hitlist
prefix: 1.0.0.0 target hitlist: ['1.0.0.0', '1.0.0.1', '1.0.0.2']
prefix: 1.0.4.0 target hitlist: ['1.0.4.1', '1.0.4.4']
prefix: 1.0.5.0 target hitlist: ['1.0.5.1', '1.0.5.5']
prefix: 1.0.6.0 target hitlist: ['1.0.6.1', '1.0.6.6']
prefix: 1.0.7.0 target hitlist: ['1.0.7.1', '1.0.7.7']
prefix: 1.0.16.0 target hitlist: ['1.0.16.14', '1.0.16.9', '1.0.16.10', '1.0.16.11']
prefix: 1.0.64.0 target hitlist: ['1.0.64.25', '1.0.64.94', '1.0.64.95']
prefix: 1.0.65.0 target hitlist: ['1.0.65.6', '1.0.65.176', '1.0.65.243']
prefix: 1.0.66.0 target hitlist: ['1.0.66.10', '1.0.66.13', '1.0.66.205']
prefix: 1.0.67.0 target hitlist: ['1.0.67.15', '1.0.67.23', '1.0.67.43']
prefix: 1.0.68.0 target hitlist: ['1.0.68.21', '1.0.68.68', '1.0.68.131']


## Build pairwise matrix
WARNING : Time consumming section

This matrix represents the geographical distance between all the probes and anchors of the dataset

下面的代码生成了一个距离矩阵，该矩阵表示了数据集中所有探针和锚点之间的地理距离。
这个矩阵的生成过程非常耗时，请耐心等待。

In [33]:
probes = load_json(USER_PROBES_FILE)
anchors = load_json(USER_ANCHORS_FILE)
anchors_ip_list = [anchor["address_v4"] for anchor in anchors]
probes.extend(anchors)

In [34]:
vp_coordinates_per_ip = {} # 键为探针或锚点的 IPv4 地址，值为对应的经纬度坐标。

for probe in probes:
    ip_v4_address = probe["address_v4"]
    long, lat = probe["geometry"]["coordinates"]
    vp_coordinates_per_ip[ip_v4_address] = lat, long

# vp_coordinates_per_ip 应当包含所有的探针和锚点的坐标。

vp_distance_matrix = {}
# vp_coordinates_per_ip_l: 按 IPv4 地址排序后的坐标列表。
# 使用 sorted 函数对 vp_coordinates_per_ip 字典的项进行排序，排序依据是 IPv4 地址。
vp_coordinates_per_ip_l = sorted(vp_coordinates_per_ip.items(), key=lambda x: x[0])

for i in range(len(vp_coordinates_per_ip_l)):
    vp_i, vp_i_coordinates = vp_coordinates_per_ip_l[i]
    if vp_i not in anchors_ip_list: # 剔除probes 探针
        continue
    for j in range(len(vp_coordinates_per_ip_l)):
        vp_j, vp_j_coordinates = vp_coordinates_per_ip_l[j]
        distance = haversine(vp_i_coordinates, vp_j_coordinates)
        vp_distance_matrix.setdefault(vp_i, {})[vp_j] = distance
        vp_distance_matrix.setdefault(vp_j, {})[vp_i] = distance


dump_json(vp_distance_matrix, USER_PAIRWISE_DISTANCE_FILE)

## Find wrongly geolocated probes （跳过此步骤）

When looking at all the pings between probes, we sometimes notice violations of the speed of Internet, which means that the corresponding probes are wrongly geolocated.  
The following algorithm removes greedily the probe with the most violations until there is none.

当查看所有探针之间的ping时，我们有时会注意到速度限制违规行为，这意味着相应探针被错误地定位。  
以下算法通过贪婪地删除违规探针直到没有违规探针。

1. 计算每个探针的违规次数。
2. 按照违规次数排序。
3. 贪婪地删除违规次数最多的探针。
4. 重复步骤3，直到没有违规探针。

## Remove bad results

Then the probes are removed of the dataset.  
We also remove anchors that haven't enough traceroute data to be analyzed.

首先，我们删除数据集中的不良结果。  
我们还删除了没有足够的traceroute数据来分析的锚点。


## Create removed and filtered probes files

All the probes removed from the beggining of the notebook are saved in the removed_probes.json file.  

All the probes that are in the clickhouse database but not in the current dataset are saved in the filtered_probes.json file.  
Later, they will be used to create a filter and collect only relevant data.



在notebook的开头，我们已经将所有被移除的探针保存到了removed_probes.json文件中。

而在clickhouse数据库中但不在当前数据集中的探针则被保存到了filtered_probes.json文件中。

之后，我们将使用这个文件来创建过滤器，只收集相关的数据。


## Select greedy probes
WARNING : Time consumming section

Greedily compute the probe with the greatest distance to other probes

翻译
贪婪地计算距离其他探针最远的探针

In [35]:
def greedy_selection_probes_impl(probe, distance_per_probe, selected_probes):

    distances_log = [math.log(distance_per_probe[p]) for p in selected_probes
                     if p in distance_per_probe and distance_per_probe[p] > 0]
    total_distance = sum(distances_log)
    return probe, total_distance

In [36]:
# vp_distance_matrix = load_json(PAIRWISE_DISTANCE_FILE)
vp_distance_matrix = load_json(USER_PAIRWISE_DISTANCE_FILE)

In [37]:
print("Starting greedy algorithm")
selected_probes = []
remaining_probes = set(vp_distance_matrix.keys())
with Pool(12) as p:
    while len(remaining_probes) > 0 and len(selected_probes) < LIMIT:
        args = []
        for probe in remaining_probes:
            args.append((probe, vp_distance_matrix[probe], selected_probes))

        results = p.starmap(greedy_selection_probes_impl, args)

        furthest_probe_from_selected, _ = max(results, key=lambda x:x[1])
        selected_probes.append(furthest_probe_from_selected)
        remaining_probes.remove(furthest_probe_from_selected)

# dump_json(selected_probes, GREEDY_PROBES_FILE)
dump_json(selected_probes, USER_GREEDY_PROBES_FILE)

Starting greedy algorithm
